## Inputs 

In [ ]:
# Email of the User.
email_user = 'mohammad.x.yunus@us.tel.com'
# Repository Path
enterprise_repository = r"C:\Users\223163\Documents\master_repo\EnterpriseCloud"
# New arch Schema
new_arch_schema = 'tmea_supply_chain'

#### How to Use
- Add your email
- Provide the schema of the new architecture you are working on.
- Input the path to your enterprise repository.

#### Run the script
Use the "Run All" option to execute the entire notebook.

#### Note:
`Before adding descriptions, make sure you have your model.yml file present in the bronze folder.`

## Imports and Utility Functions

In [41]:
from pathlib import Path
from yaml import safe_load, dump, SafeDumper
from contextlib import contextmanager
import copy
import re
import pandas as pd
import warnings
import os
import yaml
import snowflake.connector

warnings.filterwarnings('ignore')


pattern = r'''(?i)\bFROM\b\s+\{\{\s*source\(\s*["']([^"']+)["']\s*,\s*["']([^"']+)["']\s*\)\s*\}\}'''
base_path = Path(enterprise_repository).joinpath('astro_datautility','dags','dbt','dbt_tel')
_scr_path = base_path / 'models' /  '_src'

In [42]:
def find_files(root_dir, directory, file_name):
    return [
        (file / file_name).resolve()
        for file in  Path(root_dir).rglob(directory)
        if file.is_dir() and (file / file_name).is_file()
    ]

def find_all_files_in_bronze_dirs():
    return [
        file.resolve()
        for bronze_dir in Path(enterprise_repository).rglob("bronze")
        if bronze_dir.is_dir()
        for file in bronze_dir.iterdir()
        if file.is_file()
    ]

@contextmanager
def open_files(paths, mode='r+', encoding='utf-8'):
    files = []
    try:
        for path in paths:
            fp = open(path, mode, encoding=encoding)
            content = safe_load(fp.read())
            fp.seek(0)
            files.append((fp,content))
        yield files
    finally:
        for fp,_ in files:
            fp.close()

def is_english(text):
    try:
        text.encode('ascii')
        return True
    except UnicodeEncodeError:
        return False

def search_in_src_filename(match):
    for item in os.listdir(_scr_path):
        with open(os.path.join(_scr_path,item),'r') as f:
            yaml_dict = yaml.safe_load(f.read()).get('sources')
            for schema in yaml_dict:
                if schema.get('name').lower() == match.group(1).lower():
                    wanted_schema = schema.get('schema').upper()
                    for dx in schema.get('tables'):
                        if dx.get('name').lower() == match.group(2).lower():
                            wanted_table = dx.get('identifier').upper()
                            return wanted_schema,wanted_table

def search_file_name(list_of_models, file):
    return next(
        item for item in list_of_models
        if re.search(
            pattern = f'{file}',
            string =item
        )
    )

In [43]:
connection_details = {
    'user' : email_user,
    'account' : "telus-global.privatelink",
    'client_session_keep_alive' : False,
    'warehouse' : "DEVELOPER_WH",
    'database' : 'BRONZE',
    'role' : 'SYSADMIN_DEV',
    'schema' : 'PUBLIC',
    'authenticator' : "externalbrowser"
}

connection = snowflake.connector.connect(**connection_details)
model_yaml = find_files(enterprise_repository, f'{new_arch_schema}/bronze' , f'_models.yml')

## CRM Descriptions

In [44]:
crm_columns = pd.read_excel('MetaData\Crm_Attributes.xlsx')
crm_table_descriptions = pd.read_excel('MetaData\Crm_Table_Descriptions.xlsx')

In [45]:
def get_table_description(search_table):
    return {
        k: v
        for k, v in dict(crm_table_descriptions[crm_table_descriptions['LogicalName'] == search_table.lower()].iloc[:, [0,3]].values).items()
        if pd.notna(v)
    }.get(search_table.lower())

def get_column_mapping_crm(search_table):
    return {
        k: v
        for k, v in dict(crm_columns[crm_columns['EntityLogicalName'] == search_table.lower()].iloc[:, [2, 10]].values).items()
        if pd.notna(v)
    }

In [46]:
def fetch_columns_in_wherehouse(crm_model_name, schema, db ='ODS'):
    query = f"SELECT * FROM {db}.INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = '{schema.upper()}' AND TABLE_NAME = '{crm_model_name.upper()}';"
    return [
        item.lower()
        for item in pd.read_sql(query,connection).COLUMN_NAME.to_list()
    ]

In [47]:
with open_files(model_yaml) as file:
    for fp, content in file:
        temp_content = copy.deepcopy(content)
        change_made = False

        for i, item in enumerate(temp_content.get('models', [])):
            if re.search(r'CRM_FILE', item.get('name', '')):
                crm_model_name_match = re.search(r'CRM_FILE_(.*)', item.get('name', ''))
                if not crm_model_name_match:
                    continue

                crm_model_name = crm_model_name_match.group(1)
                change_made = True
                description = get_table_description(crm_model_name)
                if description and not is_english(description):
                    description = None

                column_mapping = get_column_mapping_crm(crm_model_name)
                new_columns = []

                if columns_in_warehouse := fetch_columns_in_wherehouse(crm_model_name, 'CRM_FILE'):
                    for column in columns_in_warehouse:
                        if column in column_mapping:
                            new_columns.append({
                                'name': column.upper(),
                                'description': column_mapping[column]
                            })

                existing_columns = item.get('columns', [])
                column_dict = {col['name']: col for col in existing_columns}

                for new_col in new_columns:
                    column_dict[new_col['name']] = new_col

                new_item = {}
                new_item['name'] = item['name']

                if description:
                    new_item['description'] = description
                if 'tests' in item:
                    new_item['tests'] = item['tests']
                if column_dict:
                    new_item['columns'] = list(column_dict.values())

                for key in item:
                    if key not in new_item:
                        new_item[key] = item[key]

                temp_content['models'][i] = new_item

        if change_made:
            fp.seek(0)
            fp.truncate()
            dump(temp_content, fp, sort_keys=False, allow_unicode=True, Dumper=SafeDumper)

## SAPAPI Description

In [48]:
sapapi_columns = pd.read_excel('MetaData\Sapai_Metadata.xlsx')
sapapi_descriptions = pd.read_excel('MetaData\Sapapi_Descriptions.xlsx')
sapapi_descriptions = sapapi_descriptions.applymap(lambda x: x.strip() if isinstance(x, str) else x)
sapapi_descriptions.columns = sapapi_descriptions.columns.str.strip()

In [49]:
def get_column_mapping_sapapi(search_table):
    return {
        k.lower(): v
        for k, v in dict(sapapi_columns[sapapi_columns['TABNAME'] == search_table].iloc[:, [1, 6]].values).items()
        if pd.notna(v)
    }

def get_table_description_sapapi(search_table):
    return {
        k : v
        for k, v in dict(sapapi_descriptions[sapapi_descriptions['TABNAME'] == search_table].iloc[:, [0, 1]].values).items()
        if pd.notna(v)
    }.get(search_table)

In [50]:
with open_files(model_yaml) as file:
    for fp, content in file:
        temp_content = copy.deepcopy(content)
        change_made = False

        for i, item in enumerate(temp_content.get('models', [])):
            if re.search(r'SAPAPI', item.get('name', '')):
                sapapi_model_name_match = re.search(r'SAPAPI_ZCDV_T_CMTU(.*)', item.get('name', ''))
                if not sapapi_model_name_match:
                    continue
                sapapi_model_name = sapapi_model_name_match.group(1)
                find_column_name = sapapi_model_name.split('_', maxsplit=1)[-1].replace('HIST','').replace('SECT','').replace('_','')
                change_made = True
                description = get_table_description_sapapi(find_column_name)

                if description and not is_english(description):
                    description = None
                column_mapping = get_column_mapping_sapapi(find_column_name)

                new_columns = []

                if columns_in_warehouse := fetch_columns_in_wherehouse('ZCDV_T_CMTU'+sapapi_model_name, 'SAPAPI'):

                    for column in columns_in_warehouse:
                        if column in column_mapping:
                            new_columns.append({
                                'name': column.upper(),
                                'description': column_mapping[column]
                            })

                existing_columns = item.get('columns', [])
                column_dict = {col['name']: col for col in existing_columns}

                for new_col in new_columns:
                    column_dict[new_col['name']] = new_col

                new_item = {}
                new_item['name'] = item['name']

                if description:
                    new_item['description'] = description

                if 'tests' in item:
                    new_item['tests'] = item['tests']

                if column_dict:
                    new_item['columns'] = list(column_dict.values())

                for key in item:
                    if key not in new_item:
                        new_item[key] = item[key]

                temp_content['models'][i] = new_item

        if change_made:
            fp.seek(0)
            fp.truncate()
            dump(temp_content, fp, sort_keys=False, allow_unicode=True, Dumper=SafeDumper)

## SAPO Description

In [51]:
query_column = '''
SELECT
    DD03L.FIELDNAME,
    DD04T.DDTEXT AS field_name
FROM
    LZ_HVR.sap_oracle_live.DD02L AS DD02L
LEFT JOIN
    LZ_HVR.sap_oracle_live.DD02T AS D002T
    ON DD02L.TABNAME = D002T.TABNAME
LEFT JOIN
    LZ_HVR.sap_oracle_live.DD03L AS DD03L
    ON DD02L.TABNAME = DD03L.TABNAME
LEFT JOIN
    LZ_HVR.sap_oracle_live.DD04T AS DD04T
    ON DD03L.ROLLNAME = DD04T.ROLLNAME
WHERE
    (DD04T.DDLANGUAGE = 'E' OR DD04T.DDLANGUAGE IS NULL)
    AND (DD03L.COMPTYPE = 'E' OR DD03L.COMPTYPE IS NULL)
    AND D002T.DDLANGUAGE = 'E'
    AND DD03L.AS4LOCAL = 'A'
    AND DD02L.TABCLASS = 'TRANSP'
    AND DD02L.TABNAME = '{}';
'''
query_tables = '''
SELECT DISTINCT
    DD02L.TABNAME AS TABNAME,
    D002T.DDTEXT AS tabname_txt
FROM
    LZ_HVR.sap_oracle_live.DD02L AS DD02L
LEFT JOIN
    LZ_HVR.sap_oracle_live.DD02T AS D002T
    ON DD02L.TABNAME = D002T.TABNAME
LEFT JOIN
    LZ_HVR.sap_oracle_live.DD03L AS DD03L
    ON DD02L.TABNAME = DD03L.TABNAME
LEFT JOIN
    LZ_HVR.sap_oracle_live.DD04T AS DD04T
    ON DD03L.ROLLNAME = DD04T.ROLLNAME
WHERE
    (DD04T.DDLANGUAGE = 'E' OR DD04T.DDLANGUAGE IS NULL)
    AND (DD03L.COMPTYPE = 'E' OR DD03L.COMPTYPE IS NULL)
    AND D002T.DDLANGUAGE = 'E'
    AND DD03L.AS4LOCAL = 'A'
    AND DD02L.TABCLASS = 'TRANSP'
    AND DD02L.TABNAME = '{}';
'''

def get_fqn_and_query(text):

    splits = re.split(
        pattern = r'__',
        string = text
    )

    return splits[-1],splits[-2],splits[-3]

def get_column_mapping_sapo(search_table):
    df_temp = pd.read_sql(query_column.format(search_table),connection)
    return {
        k.lower(): v
        for k, v in dict(df_temp.iloc[:, [0, 1]].values).items()
        if pd.notna(v)
    }

def get_table_description_sapo(search_table):
    df_temp = pd.read_sql(query_column.format(search_table),connection)
    return {
        k : v
        for k, v in dict(df_temp.iloc[:, [0, 1]].values).items()
        if pd.notna(v)
    }.get(search_table)

In [52]:
list_of_models = [str(item) for item in find_all_files_in_bronze_dirs()]

with open_files(model_yaml) as file:
    for fp, content in file:
        temp_content = copy.deepcopy(content)
        change_made = False

        for i, item in enumerate(temp_content.get('models', [])):
            if re.search(r'SAPO', item.get('name', '')):
                sapo_model_name_match = re.search(r'SAPO_(.*)', item.get('name', ''))
                path_of_model = search_file_name(list_of_models,item.get('name'))

                with open(path_of_model,'r') as f:
                    data_in_model = f.read()

                match = re.search(pattern, data_in_model)
                if match:
                    change_made = True
                    description = get_table_description_sapo(match.group(2))

                    if description and not is_english(description):
                        description = None
                    column_mapping = get_column_mapping_sapo(match.group(2))

                    new_columns = []

                    if columns_in_warehouse:= fetch_columns_in_wherehouse(*get_fqn_and_query(item.get('name'))):

                        for column in columns_in_warehouse:
                            if column in column_mapping:
                                new_columns.append({
                                    'name': column.upper(),
                                    'description': column_mapping[column]
                                })

                    existing_columns = item.get('columns', [])
                    column_dict = {col['name']: col for col in existing_columns}

                    for new_col in new_columns:
                        column_dict[new_col['name']] = new_col

                    new_item = {}
                    new_item['name'] = item['name']

                    if description:
                        new_item['description'] = description

                    if 'tests' in item:
                        new_item['tests'] = item['tests']

                    if column_dict:
                        new_item['columns'] = list(column_dict.values())

                    for key in item:
                        if key not in new_item:
                            new_item[key] = item[key]

                    temp_content['models'][i] = new_item

        if change_made:
            fp.seek(0)
            fp.truncate()
            dump(temp_content, fp, sort_keys=False, allow_unicode=True, Dumper=SafeDumper)